# 🎬 P36 - Script2Video AI: Unit 3
## AI Image Generation (Stable Diffusion 1.5) & BLIP Caption Verification

**Project:** P36 — Script-to-Narrated Video Maker (`PS36_PratikGholap`)

**Goal:** Generate 6 scene images using Stable Diffusion 1.5, caption them using Salesforce BLIP, calculate semantic cosine similarity with Sentence Transformers (`all-MiniLM-L6-v2`), and package all assets into `P36_U3_Output.zip` for the local Streamlit application.

### Pipeline Architecture:
1. **Input:** `video_script.json` (generated in U2 with 6 scenes).
2. **Image Generation:** `stable-diffusion-v1-5/stable-diffusion-v1-5` (FP16, memory-optimized attention slicing, deterministic seeds).
3. **Image Captioning:** `Salesforce/blip-image-captioning-base` (`BlipProcessor` + `BlipForConditionalGeneration`).
4. **Semantic Verification:** `sentence-transformers/all-MiniLM-L6-v2` calculating raw cosine similarity between intended visual description and BLIP caption.
5. **Export:** Bundle `images/`, `generation_results.json`, `caption_results.json`, and `verification_results.json` into `P36_U3_Output.zip`.

### Cell 1: Install Dependencies
Install Hugging Face Diffusers, Transformers, Accelerate, Sentence-Transformers, and PyTorch.

In [ ]:
# Cell 1: Install dependencies
!pip install -q --upgrade diffusers transformers accelerate sentence-transformers pillow torch

### Cell 2: System & GPU Verification
Verify CUDA availability, display GPU model and allocated memory.

In [ ]:
# Cell 2: Verify CUDA & GPU
import os
import sys
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")

cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU Detected: {gpu_name}")
    print(f"Total GPU Memory: {total_mem:.2f} GB")
else:
    device = "cpu"
    print("WARNING: No GPU detected. Execution will be slow on CPU. In Google Colab, go to Runtime > Change runtime type > Select T4 GPU.")

### Cell 3: Load Video Script JSON
Upload your `video_script.json` generated from U2 in the Streamlit app, or use the embedded default script.

In [ ]:
# Cell 3: Load or Upload video_script.json
import json
from pathlib import Path

script_path = Path("video_script.json")

if not script_path.exists():
    print("Choose how to load video_script.json:")
    try:
        from google.colab import files
        print("Please upload your video_script.json downloaded from Streamlit U2:")
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn.endswith(".json"):
                script_path = Path(fn)
                break
    except Exception as e:
        print(f"Could not open Colab file uploader: {e}")

# Fallback default script if no file uploaded
if not script_path.exists():
    print("Using embedded default video script (The Future of Artificial Intelligence)...")
    default_script = {
        "video_title": "The Future of Artificial Intelligence: Beyond the Horizon",
        "topic": "The Future of Artificial Intelligence",
        "total_duration": 45,
        "scenes": [
            {
                "scene_number": 1,
                "scene_title": "The Dawn of a New Era",
                "narration": "Artificial intelligence is no longer just lines of code; it is rapidly becoming the catalyst reshaping human civilization.",
                "visual_description": "A close-up of a human eye reflecting glowing, intricate neural network pathways that branch into a panoramic view of an advanced metropolis at sunrise.",
                "image_prompt": "Cinematic wide shot of glowing digital neural networks overlaying a futuristic skyline at dawn, photorealistic, volumetric golden morning lighting, 8k resolution, cinematic atmosphere.",
                "duration_seconds": 7
            },
            {
                "scene_number": 2,
                "scene_title": "Revolutionizing Medicine",
                "narration": "In healthcare, advanced algorithms are decoding complex genetics in seconds, engineering bespoke treatments that can save millions of lives.",
                "visual_description": "A compassionate doctor wearing holographic glasses interacts with a floating three-dimensional digital DNA strand inside a modern hospital laboratory.",
                "image_prompt": "Cinematic medium shot of a healthcare scientist in a high-tech lab interacting with a translucent, glowing blue holographic DNA helix, clean aesthetic, soft ambient lighting, cinematic depth of field.",
                "duration_seconds": 8
            },
            {
                "scene_number": 3,
                "scene_title": "Autonomous and Sustainable Living",
                "narration": "Autonomous systems are optimizing our cities, managing renewable energy grids and purifying environments in real time.",
                "visual_description": "Sleek magnetic levitation transit glides silently through a lush, solarpunk city filled with vertical forest skyscrapers and solar crystal bridges.",
                "image_prompt": "Cinematic shot of a solarpunk futuristic city with vertical gardens, sleek maglev trains on transparent tracks, glowing energy conduits, crystal clear skies, ultra-detailed architectural design.",
                "duration_seconds": 7
            },
            {
                "scene_number": 4,
                "scene_title": "The Evolution of Creativity",
                "narration": "The creative frontier is expanding, as human ingenuity pairs with generative systems to craft art, architecture, and stories once unimaginable.",
                "visual_description": "An artist in a futuristic studio gestures into the air, synthesizing surreal kaleidoscopic visual landscapes composed of light and sound.",
                "image_prompt": "Cinematic dramatic lighting shot of a diverse artist conducting glowing holographic paint strokes and fractal light waves in a modern darkened studio, artistic masterpiece, vivid colors.",
                "duration_seconds": 8
            },
            {
                "scene_number": 5,
                "scene_title": "The Ethical Frontier",
                "narration": "Yet this immense power demands stewardship, guiding us to forge ethical frameworks that safeguard human dignity and truth.",
                "visual_description": "Two translucent glowing hands, one mechanical and one biological, gently converge over a core of radiant blue light symbolizing balance and ethics.",
                "image_prompt": "Cinematic close-up of a robotic hand and a human hand gently touching, glowing crystalline light pulsating between fingertips, dramatic dark studio background, profound emotional lighting.",
                "duration_seconds": 7
            },
            {
                "scene_number": 6,
                "scene_title": "The Symbiotic Horizon",
                "narration": "The future of AI is not about replacing humanity, but empowering us to explore the cosmos and discover our infinite potential.",
                "visual_description": "A diverse group of explorers look outward from an orbital station viewing deck toward deep space, alongside friendly glowing robotic companions.",
                "image_prompt": "Cinematic wide establishing shot of human astronauts and gentle sleek companion robots standing side by side looking at a vibrant colorful nebula through a giant spaceship window, epic sense of wonder.",
                "duration_seconds": 8
            }
        ]
    }
    with open("video_script.json", "w", encoding="utf-8") as f:
        json.dump(default_script, f, indent=4)
    script_data = default_script
else:
    with open(script_path, "r", encoding="utf-8") as f:
        script_data = json.load(f)

print(f"Loaded script: '{script_data.get('video_title')}' with {len(script_data.get('scenes', []))} scenes.")

### Cell 4: Initialize Stable Diffusion 1.5 Pipeline
Loads `stable-diffusion-v1-5/stable-diffusion-v1-5` with FP16 precision and memory optimizations (`enable_attention_slicing`, `enable_vae_slicing`).

In [ ]:
# Cell 4: Initialize Stable Diffusion 1.5 with Memory Optimization
from diffusers import StableDiffusionPipeline
import torch

model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
print(f"Loading Stable Diffusion model: {model_id}...")

if torch.cuda.is_available():
    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        safety_checker=None  # Disable to save memory and avoid false positives on artistic scenes
    )
    pipe = pipe.to("cuda")
    # Enable memory optimizations for lower VRAM footprints
    pipe.enable_attention_slicing()
    try:
        pipe.enable_vae_slicing()
    except Exception:
        pass
    print("Stable Diffusion 1.5 loaded in FP16 with attention and VAE slicing enabled on CUDA.")
else:
    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float32,
        safety_checker=None
    )
    pipe = pipe.to("cpu")
    pipe.enable_attention_slicing()
    print("Stable Diffusion 1.5 loaded in FP32 on CPU.")

### Cell 5: Generate Six Scene Images
Iterates through the scenes, generating one image per scene with deterministic seeds and error resilience.

In [ ]:
# Cell 5: Generate Six Scene Images
import os
from pathlib import Path
from PIL import Image

output_base = Path("P36_U3_Output")
images_dir = output_base / "images"
images_dir.mkdir(parents=True, exist_ok=True)

scenes = script_data.get("scenes", [])
generation_results = []

# Standard cinematic generation settings
WIDTH = 768
HEIGHT = 432  # 16:9 ratio suitable for SD 1.5 (divisible by 8)
NUM_STEPS = 30
GUIDANCE_SCALE = 7.5
BASE_SEED = 42

print(f"Starting image generation for {len(scenes)} scenes...")

for idx, scene in enumerate(scenes, start=1):
    scene_num = scene.get("scene_number", idx)
    scene_title = scene.get("scene_title", f"Scene {scene_num}")
    prompt = scene.get("image_prompt") or scene.get("visual_description") or "Cinematic scene"
    
    # Enhanced prompt for cinematic styling
    refined_prompt = (
        f"{prompt}, cinematic composition, 16:9 aspect ratio, "
        "photorealistic, sharp details, volumetric lighting, high quality"
    )
    negative_prompt = "blurry, low quality, distorted, text, watermark, logo, deformed limbs, artifacts"
    
    image_filename = f"scene_{scene_num:02d}.png"
    image_path = images_dir / image_filename
    
    result_record = {
        "scene_number": scene_num,
        "scene_title": scene_title,
        "image_filename": image_filename,
        "image_path": str(image_path),
        "prompt_used": refined_prompt,
        "seed": BASE_SEED + scene_num,
        "status": "Failed",
        "error": None
    }
    
    print(f"\n[Scene {scene_num}/6] Generating: '{scene_title}'...")
    
    try:
        # Deterministic generator
        generator = torch.Generator(device=device).manual_seed(BASE_SEED + scene_num)
        
        # Memory fallback if OOM occurs
        try:
            image = pipe(
                prompt=refined_prompt,
                negative_prompt=negative_prompt,
                width=WIDTH,
                height=HEIGHT,
                num_inference_steps=NUM_STEPS,
                guidance_scale=GUIDANCE_SCALE,
                generator=generator
            ).images[0]
        except torch.cuda.OutOfMemoryError:
            print(f"CUDA OOM on scene {scene_num}. Retrying with reduced resolution (512x512) and 25 steps...")
            torch.cuda.empty_cache()
            image = pipe(
                prompt=refined_prompt,
                negative_prompt=negative_prompt,
                width=512,
                height=512,
                num_inference_steps=25,
                guidance_scale=GUIDANCE_SCALE,
                generator=generator
            ).images[0]
        
        image.save(image_path)
        result_record["status"] = "Generated"
        print(f"✓ Successfully generated and saved to {image_filename}")
        
    except Exception as err:
        result_record["error"] = str(err)
        print(f"✗ Failed to generate scene {scene_num}: {err}")
    
    generation_results.append(result_record)

success_count = sum(1 for r in generation_results if r["status"] == "Generated")
print(f"\nImage generation complete: {success_count}/{len(scenes)} successful.")

### Cell 6: Load BLIP & Generate Image Captions
Uses `Salesforce/blip-image-captioning-base` to generate a caption for each successfully generated scene image.

In [ ]:
# Cell 6: BLIP Image Captioning
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

blip_model_id = "Salesforce/blip-image-captioning-base"
print(f"Loading BLIP captioning model: {blip_model_id}...")

processor = BlipProcessor.from_pretrained(blip_model_id)
blip_model = BlipForConditionalGeneration.from_pretrained(blip_model_id)
blip_model = blip_model.to(device)
print("BLIP model loaded successfully.")

caption_results = []

for gen_res in generation_results:
    scene_num = gen_res["scene_number"]
    image_file = gen_res["image_filename"]
    image_full_path = Path(gen_res["image_path"])
    
    caption_record = {
        "scene_number": scene_num,
        "image_filename": image_file,
        "blip_caption": None,
        "status": "Failed",
        "error": None
    }
    
    if gen_res["status"] != "Generated" or not image_full_path.exists():
        caption_record["error"] = f"Image not available: {gen_res.get('error') or 'File missing'}"
        caption_results.append(caption_record)
        continue
    
    try:
        raw_image = Image.open(image_full_path).convert("RGB")
        inputs = processor(raw_image, return_tensors="pt").to(device)
        
        with torch.no_grad():
            out = blip_model.generate(**inputs, max_new_tokens=50)
            caption = processor.decode(out[0], skip_special_tokens=True).strip()
        
        caption_record["blip_caption"] = caption
        caption_record["status"] = "Success"
        print(f"[Scene {scene_num}] BLIP Caption: '{caption}'")
    except Exception as err:
        caption_record["error"] = str(err)
        print(f"[Scene {scene_num}] BLIP failed: {err}")
        
    caption_results.append(caption_record)

### Cell 7: Semantic Cosine Similarity Verification
Calculates semantic cosine similarity between the original visual description and the BLIP-generated caption using `sentence-transformers/all-MiniLM-L6-v2`.

> **Note on Cosine Similarity & Heuristic Threshold:**
> Cosine similarity values range from -1.0 to +1.0. We report raw cosine similarity scores directly without clamping or describing them as classification accuracy. A configurable initial review threshold (default 0.40) is clearly labeled as an exploratory heuristic.

In [ ]:
# Cell 7: Semantic Verification with Sentence Transformers
from sentence_transformers import SentenceTransformer, util

sim_model_id = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading sentence similarity model: {sim_model_id}...")
sim_model = SentenceTransformer(sim_model_id, device=device)

# Configurable initial heuristic review threshold
HEURISTIC_THRESHOLD = 0.40

verification_results = []

for idx, scene in enumerate(scenes, start=1):
    scene_num = scene.get("scene_number", idx)
    scene_title = scene.get("scene_title", f"Scene {scene_num}")
    expected_desc = scene.get("visual_description", "").strip()
    
    cap_entry = next((c for c in caption_results if c["scene_number"] == scene_num), None)
    caption = cap_entry.get("blip_caption") if cap_entry else None
    image_filename = cap_entry.get("image_filename") if cap_entry else f"scene_{scene_num:02d}.png"
    
    v_record = {
        "scene_number": scene_num,
        "scene_title": scene_title,
        "original_visual_description": expected_desc,
        "image_filename": image_filename,
        "blip_caption": caption,
        "cosine_similarity": None,
        "heuristic_threshold": HEURISTIC_THRESHOLD,
        "verification_status": "Needs Review (Heuristic)",
        "manual_review_status": "Pending",
        "manual_notes": "",
        "error": None
    }
    
    if not caption:
        v_record["error"] = cap_entry.get("error") if cap_entry else "No caption produced"
        verification_results.append(v_record)
        continue
    
    try:
        emb_expected = sim_model.encode(expected_desc, convert_to_tensor=True)
        emb_caption = sim_model.encode(caption, convert_to_tensor=True)
        
        # Calculate raw cosine similarity (-1.0 to 1.0)
        raw_cos_sim = float(util.cos_sim(emb_expected, emb_caption).item())
        cos_sim_score = round(raw_cos_sim, 4)
        v_record["cosine_similarity"] = cos_sim_score
        
        if cos_sim_score >= HEURISTIC_THRESHOLD:
            v_record["verification_status"] = "Relevant"
        else:
            v_record["verification_status"] = "Needs Review (Heuristic)"
            
        print(f"[Scene {scene_num}] Cosine Sim: {cos_sim_score:+.4f} | Status: {v_record['verification_status']}")
    except Exception as err:
        v_record["error"] = str(err)
        print(f"[Scene {scene_num}] Similarity failed: {err}")
        
    verification_results.append(v_record)

### Cell 8: Save JSON Metadata Reports
Saves `generation_results.json`, `caption_results.json`, and `verification_results.json` into `P36_U3_Output/`.

In [ ]:
# Cell 8: Save JSON Reports
import json

output_base = Path("P36_U3_Output")
output_base.mkdir(parents=True, exist_ok=True)

# 1. generation_results.json
with open(output_base / "generation_results.json", "w", encoding="utf-8") as f:
    json.dump(generation_results, f, indent=4)

# 2. caption_results.json
with open(output_base / "caption_results.json", "w", encoding="utf-8") as f:
    json.dump(caption_results, f, indent=4)

# 3. verification_results.json
with open(output_base / "verification_results.json", "w", encoding="utf-8") as f:
    json.dump(verification_results, f, indent=4)

print("All metadata reports successfully written to P36_U3_Output/:")
print(" - generation_results.json")
print(" - caption_results.json")
print(" - verification_results.json")

### Cell 9: Package Output into P36_U3_Output.zip and Download
Creates `P36_U3_Output.zip` and triggers automatic download in Google Colab.

In [ ]:
# Cell 9: Create ZIP archive and download
import shutil
import zipfile

zip_filename = "P36_U3_Output.zip"
print(f"Creating {zip_filename}...")

# Package entire P36_U3_Output folder
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk("P36_U3_Output"):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, ".")
            zipf.write(file_path, arcname)

print(f"Successfully created {zip_filename} ({os.path.getsize(zip_filename) / (1024*1024):.2f} MB).")

# Trigger Colab download
try:
    from google.colab import files
    print("Triggering automatic browser download...")
    files.download(zip_filename)
except Exception as e:
    print(f"Not in Google Colab or download blocked: {e}")
    print(f"You can manually download '{zip_filename}' from the file explorer on the left.")

### Cell 10: Next Steps in PyCharm / Streamlit
1. Download `P36_U3_Output.zip`.
2. Open your local Streamlit app in PyCharm.
3. In section **4. U3 - AI Image Generation and BLIP Verification**, upload `P36_U3_Output.zip`.
4. Streamlit will validate the archive, extract images to `generated/images/`, and display the scene gallery, BLIP captions, cosine similarity metrics, heuristic threshold controls, and verification report download.